# Experiment Tracking with Snowflake ML

This notebook demonstrates **Snowflake Experiment Tracking**:

- Create experiments and runs to organize model training
- **Autolog** parameters and metrics for XGBoost and LightGBM
- Manually log custom metrics
- Compare runs side-by-side in Snowsight

We train 3 models and compare their performance to find the best fraud detector.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.experiment.callback.xgboost import SnowflakeXgboostCallback
from snowflake.ml.experiment.callback.lightgbm import SnowflakeLightgbmCallback
from snowflake.ml.model.model_signature import infer_signature

session = get_active_session()
session.query_tag = {"origin": "ml_deep_dive", "name": "banking_fraud_detection", "notebook": "02_experiment_tracking"}

DB = "BANKING_ML_DEMO"
SCHEMA = "FRAUD_DETECTION"
WH = "ML_DEMO_WH"

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Context: {session.get_fully_qualified_current_schema()}")

## Load Training Data from Feature Store

In [ ]:
# Load the enriched training dataset created in notebook 01
train_df = session.table("TXN_FEATURES")

# Join with customer features
cust_features = session.sql("""
    SELECT CUSTOMER_ID,
           AVG(AMOUNT) AS CUST_AVG_TRANSACTION_AMT,
           COUNT(*) AS CUST_TOTAL_TRANSACTIONS,
           STDDEV(AMOUNT) AS CUST_STDDEV_AMOUNT,
           MAX(AMOUNT) AS CUST_MAX_AMOUNT,
           COUNT(DISTINCT MERCHANT_ID) AS CUST_UNIQUE_MERCHANTS,
           AVG(IS_ONLINE) AS CUST_PCT_ONLINE,
           AVG(CUSTOMER_AGE) AS CUST_AGE,
           AVG(CUSTOMER_INCOME) AS CUST_INCOME
    FROM RAW_TRANSACTIONS
    GROUP BY CUSTOMER_ID
""")

# Join with merchant features
merch_features = session.sql("""
    SELECT MERCHANT_ID,
           AVG(AMOUNT) AS MERCH_AVG_TRANSACTION_AMT,
           COUNT(*) AS MERCH_TOTAL_TRANSACTIONS,
           AVG(IS_FRAUD) AS MERCH_FRAUD_RATE
    FROM RAW_TRANSACTIONS
    GROUP BY MERCHANT_ID
""")

# Build full feature set
full_df = train_df.join(cust_features, "CUSTOMER_ID").join(merch_features, "MERCHANT_ID")
pdf = full_df.to_pandas()

print(f"Dataset shape: {pdf.shape}")
print(f"Fraud rate: {pdf['IS_FRAUD'].mean():.2%}")

In [ ]:
# Prepare features for training
FEATURE_COLS = [
    "AMOUNT", "IS_ONLINE", "TXN_HOUR", "TXN_DAY_OF_WEEK", "TXN_IS_WEEKEND",
    "TXN_IS_LATE_NIGHT", "TXN_AMOUNT_RATIO", "TXN_IS_HIGH_AMOUNT",
    "CUST_AVG_TRANSACTION_AMT", "CUST_TOTAL_TRANSACTIONS", "CUST_STDDEV_AMOUNT",
    "CUST_MAX_AMOUNT", "CUST_UNIQUE_MERCHANTS", "CUST_PCT_ONLINE",
    "CUST_AGE", "CUST_INCOME",
    "MERCH_AVG_TRANSACTION_AMT", "MERCH_TOTAL_TRANSACTIONS", "MERCH_FRAUD_RATE"
]
LABEL_COL = "IS_FRAUD"

X = pdf[FEATURE_COLS].fillna(0).astype(float)
y = pdf[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Class imbalance ratio
scale_pos = len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Positive class weight: {scale_pos:.1f}")

## Initialize Experiment Tracking

In [ ]:
exp = ExperimentTracking(session=session)
exp.set_experiment("FRAUD_DETECTION_EXPERIMENT")

print("Experiment 'FRAUD_DETECTION_EXPERIMENT' ready")
print("Navigate to Snowsight > AI & ML > Experiments to view")

## Run 1: Baseline XGBoost

Default hyperparameters with class-weight adjustment for imbalanced data.

In [ ]:
from datetime import datetime

sig = infer_signature(X_train.iloc[:100], y_train.iloc[:100])
callback_baseline = SnowflakeXgboostCallback(
    exp, model_name="xgb_baseline", model_signature=sig
)

xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss",
    callbacks=[callback_baseline]
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with exp.start_run(f"xgb_baseline_run_{timestamp}"):
    xgb_baseline.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred_bl = xgb_baseline.predict(X_test)
    y_proba_bl = xgb_baseline.predict_proba(X_test)[:, 1]
    
    bl_accuracy = accuracy_score(y_test, y_pred_bl)
    bl_precision = precision_score(y_test, y_pred_bl)
    bl_recall = recall_score(y_test, y_pred_bl)
    bl_f1 = f1_score(y_test, y_pred_bl)
    bl_auc = roc_auc_score(y_test, y_proba_bl)
    
    exp.log_metrics({
        "accuracy": bl_accuracy,
        "precision": bl_precision,
        "recall": bl_recall,
        "f1_score": bl_f1,
        "roc_auc": bl_auc
    })
    
    print(f"Baseline XGBoost Results:")
    print(f"Accuracy:  {bl_accuracy:.4f}")
    print(f"Precision: {bl_precision:.4f}")
    print(f"Recall:    {bl_recall:.4f}")
    print(f"F1 Score:  {bl_f1:.4f}")
    print(f"ROC AUC:   {bl_auc:.4f}")

## Run 2: Tuned XGBoost

Deeper trees, lower learning rate, regularization, and subsampling.

In [ ]:
from datetime import datetime

callback_tuned = SnowflakeXgboostCallback(
    exp, model_name="xgb_tuned", model_signature=sig
)

xgb_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    eval_metric="aucpr",
    random_state=42,
    callbacks=[callback_tuned]
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with exp.start_run(f"xgb_tuned_run_{timestamp}"):
    xgb_tuned.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred_tu = xgb_tuned.predict(X_test)
    y_proba_tu = xgb_tuned.predict_proba(X_test)[:, 1]
    
    tu_auc = roc_auc_score(y_test, y_proba_tu)
    tu_f1 = f1_score(y_test, y_pred_tu)
    tu_precision = precision_score(y_test, y_pred_tu)
    tu_recall = recall_score(y_test, y_pred_tu)
    
    exp.log_metrics({
        "test_auc": tu_auc,
        "test_f1": tu_f1,
        "test_precision": tu_precision,
        "test_recall": tu_recall
    })

print(f"Tuned XGBoost - AUC: {tu_auc:.4f} | F1: {tu_f1:.4f} | Precision: {tu_precision:.4f} | Recall: {tu_recall:.4f}")
print("\n" + classification_report(y_test, y_pred_tu, target_names=["Legitimate", "Fraud"]))

## Run 3: LightGBM

In [ ]:
from datetime import datetime

callback_lgbm = SnowflakeLightgbmCallback(
    exp, model_name="lgbm_model", model_signature=sig
)

lgbm_model = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with exp.start_run(f"lgbm_run_{timestamp}"):
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[callback_lgbm]
    )
    
    y_pred_lg = lgbm_model.predict(X_test)
    y_proba_lg = lgbm_model.predict_proba(X_test)[:, 1]
    
    lg_auc = roc_auc_score(y_test, y_proba_lg)
    lg_f1 = f1_score(y_test, y_pred_lg)
    lg_precision = precision_score(y_test, y_pred_lg)
    lg_recall = recall_score(y_test, y_pred_lg)
    
    exp.log_metrics({
        "test_auc": lg_auc,
        "test_f1": lg_f1,
        "test_precision": lg_precision,
        "test_recall": lg_recall
    })

print(f"LightGBM - AUC: {lg_auc:.4f} | F1: {lg_f1:.4f} | Precision: {lg_precision:.4f} | Recall: {lg_recall:.4f}")
print("\n" + classification_report(y_test, y_pred_lg, target_names=["Legitimate", "Fraud"]))

## Compare Experiment Runs

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    "Model": ["XGBoost Baseline", "XGBoost Tuned", "LightGBM"],
    "AUC": [bl_auc, tu_auc, lg_auc],
    "F1 Score": [bl_f1, tu_f1, lg_f1],
    "Precision": [bl_precision, tu_precision, lg_precision],
    "Recall": [bl_recall, tu_recall, lg_recall],
})

print("=" * 70)
print("EXPERIMENT COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False, float_format="{:.4f}".format))
print("=" * 70)

# Determine best model
best_idx = comparison["F1 Score"].idxmax()
best_model_name = comparison.loc[best_idx, "Model"]
print(f"\nBest model (by F1): {best_model_name}")
print("\nView visual comparison in Snowsight > AI & ML > Experiments > FRAUD_DETECTION_EXPERIMENT")

In [ ]:
# Store the best model and its metrics for the next notebook
models = {
    "XGBoost Baseline": (xgb_baseline, bl_auc, bl_f1, bl_precision, bl_recall),
    "XGBoost Tuned": (xgb_tuned, tu_auc, tu_f1, tu_precision, tu_recall),
    "LightGBM": (lgbm_model, lg_auc, lg_f1, lg_precision, lg_recall),
}

best_model_obj, best_auc, best_f1, best_prec, best_rec = models[best_model_name]
print(f"Best model '{best_model_name}' ready for Model Registry (notebook 03)")
print(f"  AUC={best_auc:.4f}, F1={best_f1:.4f}, Precision={best_prec:.4f}, Recall={best_rec:.4f}")

## Next Steps

Proceed to **`03_model_registry.ipynb`** to:
- Log models to the Snowflake Model Registry
- Manage versions and metadata
- Run batch inference and model explainability (SHAP)